# Export two final master reports

Import into Databricks dev and attach Unity Catalog-enabled compute. Run All with `DRY_RUN=true` first; check the two matches, then set `DRY_RUN=false` and Run All to export.

This notebook reads `vaid.ai_std_con_field_service_report.fsr_metadata_v2` and follows each row's `parsed_volume_path` to the existing extracted JSON. Each report gets `parsed.json` (unchanged), `raw_extracted.txt`, and `metadata.json` (the full metadata row). A manifest records source paths and any text fallback. No ZIP is created, and source tables/files are not changed.

The default `EXPORT_ROOT` is `/Volumes/vaid/ai_std_con_field_service_report/fsr_v2_test/report_exports`. The notebook creates `report_exports` and a fresh timestamped subfolder automatically; the `fsr_v2_test` volume must already exist. If a Databricks widget already exists, change its value explicitly to use the new default. Required access: `SELECT` on metadata, `READ VOLUME` on the source, `WRITE VOLUME` on the destination, plus catalog/schema usage.

Raw text uses `raw_pages`; if unavailable, it uses cleaned `pages` and records that fallback in the manifest. Original PDF binaries are not included. Download the exported files from Catalog Explorer using the printed paths. Share only with authorized recipients.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F

dbutils.widgets.text("METADATA_TABLE", "vaid.ai_std_con_field_service_report.fsr_metadata_v2")
dbutils.widgets.text("EXPORT_ROOT", "/Volumes/vaid/ai_std_con_field_service_report/fsr_v2_test/report_exports")
dbutils.widgets.dropdown("DRY_RUN", "true", ["true", "false"])
metadata_table = dbutils.widgets.get("METADATA_TABLE").strip()
export_root = dbutils.widgets.get("EXPORT_ROOT").strip()
dry_run = dbutils.widgets.get("DRY_RUN") == "true"
if len(metadata_table.split(".")) != 3:
    raise ValueError("METADATA_TABLE must be catalog.schema.table")
if not export_root.startswith("/Volumes/"):
    raise ValueError("EXPORT_ROOT must be an approved /Volumes/ path")
print("Reading metadata from:", metadata_table)

report_names = [
    "93c2180b-f866-4521-b090-b88442c6dcf1_204006084-2510-297422-Final_Master_Report.pdf",
    "9ecb1354-1f3a-4634-8a74-12943aff45a9_605003354-23849-298081-Final_Master_Report.pdf",
]
reports = [{"document_id": name.split("_", 1)[0], "pdf_name": name} for name in report_names]
metadata_df = spark.table(metadata_table)
required_columns = {"document_id", "parsed_volume_path"}
if not required_columns.issubset(metadata_df.columns):
    raise ValueError(f"Missing metadata columns: {required_columns - set(metadata_df.columns)}")
matched_df = metadata_df.filter(
    F.lower(F.trim(F.col("document_id"))).isin([report["document_id"] for report in reports])
)
metadata_rows = matched_df.collect()
prepared = []
for report in reports:
    matches = [row for row in metadata_rows if str(row["document_id"]).strip().lower() == report["document_id"]]
    if len(matches) != 1:
        raise RuntimeError(f"{report['pdf_name']}: expected one metadata row, found {len(matches)} in {metadata_table}. Check the METADATA_TABLE widget; dev uses vaid.ai_std_con_field_service_report.fsr_metadata_v2.")
    metadata = matches[0].asDict(recursive=True)
    parsed_path = str(metadata.get("parsed_volume_path") or "")
    if not parsed_path.startswith("/Volumes/") or Path(parsed_path).suffix.lower() != ".json":
        raise ValueError(f"{report['document_id']}: invalid parsed_volume_path: {parsed_path!r}")
    with open(parsed_path, encoding="utf-8") as source_file:
        parsed = json.load(source_file)
    if not isinstance(parsed, dict):
        raise ValueError(f"Expected parsed JSON object: {parsed_path}")
    text_source = "raw_pages" if parsed.get("raw_pages") else "pages"
    pages = parsed.get(text_source)
    if not isinstance(pages, list) or not pages or not all(isinstance(page, str) for page in pages):
        raise ValueError(f"{parsed_path}: {text_source} must be a nonempty list of page strings")
    prepared.append({"report": report, "metadata": metadata, "parsed_path": parsed_path, "pages": pages, "text_source": text_source})

preview_columns = [column for column in ["document_id", "pdf_name", "parsed_volume_path", "metadata_status", "chunk_status"] if column in matched_df.columns]
display(matched_df.select(*preview_columns))
for item in prepared:
    print(item["report"]["document_id"], "matched;", len(item["pages"]), "pages; text source:", item["text_source"])

if dry_run:
    print("Dry run complete: both metadata rows and parsed artifacts verified. No files written. Set DRY_RUN=false and Run All to export.")
else:
    exported_at = datetime.now(timezone.utc)
    export_dir = Path(export_root) / exported_at.strftime("final_master_reports_%Y%m%dT%H%M%S_%fZ")
    export_dir.mkdir(parents=True, exist_ok=False)
    exported_files = []
    manifest_documents = []
    for item in prepared:
        report = item["report"]
        document_dir = export_dir / report["document_id"]
        document_dir.mkdir()
        parsed_output = document_dir / "parsed.json"
        text_output = document_dir / "raw_extracted.txt"
        metadata_output = document_dir / "metadata.json"
        shutil.copyfile(item["parsed_path"], parsed_output)
        text_output.write_text("\n\n".join(item["pages"]), encoding="utf-8")
        metadata_output.write_text(json.dumps(item["metadata"], indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        exported_files.extend([parsed_output, text_output, metadata_output])
        manifest_documents.append({
            "document_id": report["document_id"],
            "requested_pdf_name": report["pdf_name"],
            "stored_pdf_name": item["metadata"].get("pdf_name"),
            "source_parsed_volume_path": item["parsed_path"],
            "text_source": item["text_source"],
            "page_count": len(item["pages"]),
            "export_folder": str(document_dir),
        })
    manifest_path = export_dir / "manifest.json"
    manifest_path.write_text(json.dumps({"exported_at": exported_at.isoformat(), "metadata_table": metadata_table, "documents": manifest_documents}, indent=2), encoding="utf-8")
    exported_files.append(manifest_path)
    print("Export folder:", export_dir)
    for output_file in exported_files:
        print(str(output_file), output_file.stat().st_size, "bytes")
    print("Download the files from Catalog Explorer > fsr_v2_test > report_exports > timestamped export folder.")